In [9]:
# Import required libraries
import pandas as pd
import plotly.graph_objects as go
import dataretrieval.nwis as nwis  # USGS dataRetrieval package


In [10]:
# -------------------------------
# Data Request Parameters
# -------------------------------
site_no = "08330000"          # USGS site ID (Rio Grande at Albuquerque) - without 'USGS-' prefix
parameter_cd = "00060"        # Parameter code for discharge (cubic feet per second)
start_date = "2025-05-01"    # Start date
end_date = "2025-09-30"      # End date


In [11]:
# -------------------------------
# Fetch daily values using dataRetrieval
# -------------------------------
# get_dv() fetches daily values for the specified site and parameter
df, metadata = nwis.get_dv(
    sites=site_no,
    parameterCd=parameter_cd,
    start=start_date,
    end=end_date
)

print("Data retrieved successfully!")
print(f"Shape: {df.shape}")
print("\nFirst few rows:")
display(df.head())


Data retrieved successfully!
Shape: (153, 3)

First few rows:


,site_no,00060_Mean,00060_Mean_cd
datetime,,,
2025-05-01 00:00:00+00:00,08330000,380.0,A
2025-05-02 00:00:00+00:00,08330000,383.0,A
2025-05-03 00:00:00+00:00,08330000,374.0,A
2025-05-04 00:00:00+00:00,08330000,399.0,A
2025-05-05 00:00:00+00:00,08330000,529.0,A


In [12]:
# -------------------------------
# Data Cleaning and Preparation
# -------------------------------
# Reset index to make datetime a column
df = df.reset_index()

# Rename columns for clarity
# dataRetrieval returns columns like: datetime, site_no, 00060_Mean, 00060_Mean_cd
df = df.rename(columns={
    'datetime': 'date',
    '00060_Mean': 'value'  # The actual discharge column
})

# Keep only date and value columns
df = df[['date', 'value']]

# Add parameter_code and statistic_id for consistency
df['parameter_code'] = parameter_cd
df['statistic_id'] = '00003'  # Daily mean

# Add month column
df['month'] = df['date'].dt.month

print("Cleaned DataFrame:")
display(df.head())
print(f"\nTotal records: {len(df)}")


Cleaned DataFrame:


,date,value,parameter_code,statistic_id,month
0,2025-05-01 00:00:00+00:00,380.0,00060,00003,5
1,2025-05-02 00:00:00+00:00,383.0,00060,00003,5
2,2025-05-03 00:00:00+00:00,374.0,00060,00003,5
3,2025-05-04 00:00:00+00:00,399.0,00060,00003,5
4,2025-05-05 00:00:00+00:00,529.0,00060,00003,5



Total records: 153


In [13]:
# -------------------------------
# Save the DataFrame to a CSV file
# -------------------------------
year_label = pd.to_datetime(start_date).year
site_label = f"USGS-{site_no}"  # Add USGS prefix for consistency
csv_filename = f"{site_label}_{year_label}_daily_dataRetrieval.csv"
df.to_csv(csv_filename, index=False)
print(f"Data saved to {csv_filename}")


Data saved to USGS-08330000_2025_daily_dataRetrieval.csv


In [14]:
# -------------------------------
# Monthly Statistics
# -------------------------------
# Convert discharge values to numeric (float), invalid entries become NaN
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Drop rows where value is NaN
df = df.dropna(subset=['value'])

# Reset month column just in case
df['month'] = df['date'].dt.month

# Compute monthly stats and include parameter_code and statistic_id
monthly_stats = df.groupby('month').agg(
    parameter_code=('parameter_code', 'first'),
    statistic_id=('statistic_id', 'first'),
    min=('value', 'min'),
    max=('value', 'max'),
    mean=('value', 'mean'),
    total_discharge=('value', 'sum'),
).reset_index()

# Round ONLY the mean column to 2 decimals
monthly_stats['mean'] = monthly_stats['mean'].round(2)

# Add month name
monthly_stats['Month'] = pd.to_datetime(monthly_stats['month'], format='%m').dt.strftime('%b')

# Reorder columns (Month name, parameter_code, statistic_id, stats...)
monthly_stats = monthly_stats[[
    'Month', 'parameter_code', 'statistic_id', 'min', 'max', 'mean', 'total_discharge'
 ]]

display(monthly_stats)

# Compute overall min/max
overall_min = df['value'].min()
overall_max = df['value'].max()

# Find all months where min and max occur
min_months_list = df[df['value'] == overall_min]['date'].dt.strftime('%b').unique()
max_months_list = df[df['value'] == overall_max]['date'].dt.strftime('%b').unique()

# Join with commas if multiple months
min_months_str = ', '.join(min_months_list)
max_months_str = ', '.join(max_months_list)

print(
    f"Overall minimum discharge in {year_label} ({min_months_str}): "
    f"{overall_min:.2f} cfs"
)

print(
    f"Overall maximum discharge in {year_label} ({max_months_str}): "
    f"{overall_max:.2f} cfs"
)

# -------------------------------
# Min / Max TOTAL monthly discharge
# -------------------------------
min_total_row = monthly_stats.loc[monthly_stats['total_discharge'].idxmin()]
max_total_row = monthly_stats.loc[monthly_stats['total_discharge'].idxmax()]

print(
    f"Minimum total monthly discharge ({min_total_row['Month']}): "
    f"{min_total_row['total_discharge']:.2f} cfs"
)

print(
    f"Maximum total monthly discharge ({max_total_row['Month']}): "
    f"{max_total_row['total_discharge']:.2f} cfs"
)

,Month,parameter_code,statistic_id,min,max,mean,total_discharge
0,May,00060,00003,333.00,1010.0,558.52,17314.00
1,Jun,00060,00003,151.00,820.0,411.63,12349.00
2,Jul,00060,00003,0.00,565.0,51.83,1606.86
3,Aug,00060,00003,0.00,259.0,32.43,1005.25
4,Sep,00060,00003,0.31,365.0,103.15,3094.64


Overall minimum discharge in 2025 (Jul, Aug): 0.00 cfs
Overall maximum discharge in 2025 (May): 1010.00 cfs
Minimum total monthly discharge (Aug): 1005.25 cfs
Maximum total monthly discharge (May): 17314.00 cfs


In [20]:
# -------------------------------
# Interactive Plot with Plotly
# -------------------------------
date_col = "date"
value_column = "value"

# Find all months where min and max occur
min_value = df['value'].min()
max_value = df['value'].max()
min_months = df[df['value'] == min_value]['date'].dt.strftime('%b').unique()
max_months = df[df['value'] == max_value]['date'].dt.strftime('%b').unique()

# Join with commas if multiple months
min_month_str = ', '.join(min_months)
max_month_str = ', '.join(max_months)

fig = go.Figure()

# --- Main daily discharge line ---
fig.add_trace(
    go.Scatter(
        x=df[date_col],
        y=df[value_column],
        mode="lines+markers",
        name="Daily Discharge",
        marker=dict(size=4),
        line=dict(color="blue"),
        hovertemplate=
        "<b>Date:</b> %{x|%Y-%m-%d}<br>"
        "<b>Discharge:</b> %{y:.2f} cfs<extra></extra>"
    )
)

# --- Add annotation message above the plot area ---
fig.add_annotation(
    text=f"For Period: {start_date} to {end_date}<br>Min Discharge: {overall_min:.2f} cfs ({min_month_str})<br>Max Discharge: {overall_max:.2f} cfs ({max_month_str})",
    xref="paper",
    yref="paper",
    x=1.07, 
    y=1.08,
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    bgcolor="rgba(255,255,255,0.8)",
    bordercolor="gray",
    borderwidth=1
 )

# --- Layout settings ---
fig.update_layout(
    title=f"Interactive Daily Discharge - Site USGS-{site_no}({year_label})<br>[dataRetrieval]</span>",
    xaxis_title="Date",
    yaxis_title="Discharge (cfs)",
    hovermode="x unified",
    template="plotly_white",
    xaxis=dict(
        dtick="M1",          # Tick every month
        tickformat="%b %Y",  # Format month and year
        tickangle=-45
    ),
    height=600,
    margin=dict(t=120)
 )

fig.show()
